In [ ]:
from google import genai
from google.genai import types
import os
import json
import time
import pandas as pd
from pydantic import BaseModel, Field, ValidationError

In [ ]:
#api key generated from gemini/chatgpt or claude
api_key = os.environ.get("careloop_clinic_key")

In [ ]:
client = genai.Client(api_key=api_key)

In [ ]:
MODEL_NAME = "models/gemini-3.8-flash"

In [ ]:
ALLOWED_CAUSES = [
    "forgot_or_no_reminder",
    "distance_or_transport",
    "cost_or_fee",
    "illness_or_emergency",
    "work_or_schedule",
    "other",
]

In [ ]:
SYSTEM_TEXT = (
    "You are an expert clinical operations data extractor for CareLoop Clinics."
)

In [ ]:
prompt = f"""
[Context] The front desk logged a post-appointment follow-up call note from a patient who missed their clinic visit.
[Role] Extract structured root-cause analytics from the raw text to assist predictive no-show mitigation.
[Action] Analyse the provided call note and classify it into exactly one root cause from this allowed list: {ALLOWED_CAUSES}. If the note is ambiguous or does not fit, classify it as 'other'. Provide a confidence score between 0 and 1, a brief summary under 25 words, and a recommended front-desk action.
[Tone] Objective, clinical, and precise.
[Format] Return valid JSON adhering strictly to the response schema.

Call Note to Analyse (Note ID: N001):
"Forgot about the appointment. No reminder SMS received."
"""

In [ ]:
SCHEMA = {
    "type": "object",
    "properties": {
        "root_cause": {"type": "string"},
        "confidence": {"type": "number"},
        "summary": {"type": "string"},
        "recommended_action": {"type": "string"},
    },
    "required": [
        "root_cause",
        "confidence",
        "summary",
        "recommended_action",
    ],
}

In [ ]:
def analyse_note(note_text):
  formatted_prompt = f"""
    [Context] The front desk logged a post-appointment follow-up call note from a patient who missed their clinic visit.
    [Role] Extract structured root-cause analytics from the raw text.
    [Action] Analyse the provided call note and classify it into exactly one root cause from this allowed list: {ALLOWED_CAUSES}. If ambiguous, use 'other'. Provide confidence (0-1), a summary (<25 words), and a recommended action.
    [Tone] Objective, clinical, and precise.
    [Format] Return valid JSON adhering strictly to the response schema.

    Call Note: "{note_text}"
    """
  response = client.models.generate_content(
        model="models/gemini-3.8-flash",
        contents=prompt,
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_TEXT,
            temperature=0,
            max_output_tokens=1000,
            response_mime_type="application/json",
            response_schema=SCHEMA,
        ),
  )
  return json.loads(response.text)

In [ ]:
result_n001 = analyse_note("Forgot about the appointment. No reminder SMS received.")
print("N001 Result:", json.dumps(result_n001, indent=2))

In [ ]:
ALLOWED_SET = {
    "forgot_or_no_reminder",
    "distance_or_transport",
    "cost_or_fee",
    "illness_or_emergency",
    "work_or_schedule",
    "other",
}

In [ ]:
class FollowupInsight(BaseModel):
  root_cause: str = Field(min_length=3)
  confidence: float = Field(ge=0, le=1)
  summary: str = Field(min_length=5)
  recommended_action: str = Field(min_length=5)

In [ ]:
notes_df = pd.read_csv("data/followup_notes.csv")
analysis_records = []
api_call_count = 0

for idx, row in notes_df.head(10).iterrows():
  note_id = row["note_id"]
  appointment_id = row["appointment_id"]
  note_text = row["note_text"]

  # Check for missing, empty, or whitespace-only notes
  if pd.isna(note_text) or str(note_text).strip() == "":
    analysis_records.append({
        "note_id": note_id,
        "appointment_id": appointment_id,
        "root_cause": "other",
        "confidence": 0.0,
        "summary": "No note recorded",
        "recommended_action": "Re-call patient",
        "validation_status": "skipped_blank",
        "raw_response": "",
    })
    continue

  # Call API & increment count
  api_call_count += 1
  raw_text = ""
  try:
    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=(
            f"Analyse this note using allowed causes {list(ALLOWED_SET)}: "
            f'"{note_text}"'
        ),
       config = types.GenerateContentConfig(
          system_instruction=SYSTEM_TEXT,
          temperature=0,
          max_output_tokens=1000,
          response_mime_type="application/json",
          response_schema=SCHEMA,
      ),
    )
    raw_text = response.text
    data = json.loads(raw_text)
   
    insight = FollowupInsight(**data)
    if insight.root_cause not in ALLOWED_SET:
      raise ValueError(f"Root cause '{insight.root_cause}' not in allowed list.")

    analysis_records.append({
        "note_id": note_id,
        "appointment_id": appointment_id,
        "root_cause": insight.root_cause,
        "confidence": insight.confidence,
        "summary": insight.summary,
        "recommended_action": insight.recommended_action,
        "validation_status": "valid",
        "raw_response": raw_text,
    })

  except (ValidationError, ValueError, json.JSONDecodeError) as e:
    partial_data = {}
    try:
      partial_data = json.loads(raw_text) if raw_text else {}
    except Exception:
      pass

    analysis_records.append({
        "note_id": note_id,
        "appointment_id": appointment_id,
        "root_cause": partial_data.get("root_cause", ""),
        "confidence": partial_data.get("confidence", None),
        "summary": partial_data.get("summary", ""),
        "recommended_action": partial_data.get("recommended_action", ""),
        "validation_status": "invalid",
        "raw_response": raw_text,
    })

  time.sleep(13)

In [ ]:
notes_analysed_df = pd.DataFrame(analysis_records)
notes_analysed_df.to_csv("notes_analysed.csv", index=False)
print(f"Processed 10 notes. Total API calls made: {api_call_count}")

In [ ]:
print("--- Testing Pydantic Error Catching ---")
try:
  FollowupInsight(
      root_cause="forgot_or_no_reminder",
      confidence=1.7,  # Invalid: > 1
      summary="Patient forgot appointment date.",
      recommended_action="Call to reschedule immediately.",
  )
except ValidationError as e:
  print("Caught Confidence Error:\n", e)

try:
  FollowupInsight(
      root_cause="forgot_or_no_reminder",
      confidence=0.95,
      summary="Patient forgot appointment date.",
      # Missing recommended_action field
  )
except ValidationError as e:
  print("\nCaught Missing Field Error:\n", e)

In [ ]:
analysed = pd.read_csv("notes_analysed.csv")
cleaned = pd.read_csv("appointments_clean.csv")

merged_notes = pd.merge(analysed, cleaned, on="appointment_id", how="inner")

crosstab_result = pd.crosstab(
    merged_notes["root_cause"], merged_notes["reminder_sent"]
)
print("--- Cross-Tabulation: Root Cause vs Reminder Sent ---")
print(crosstab_result)